# 第 07 章：MoE 排序路由动手实验

本 Notebook 依次完成环境检查、工程阅读、910B 目标构建、数据生成、路由对比和完整 pipeline 分析。设备命令需要在已安装 CANN 的 Linux/NPU 环境执行。

## 1. 检查实验环境

本单元定位实验根目录和源码目录，并显示 CANN 环境变量。若当前环境尚未安装 CANN，先阅读后续单元和 `910b_guide.md`，不要把个人绝对路径写入代码。

In [ ]:
import os
from pathlib import Path

def locate_lab_dir():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src' / 'moe_sort_lab').is_dir():
            return candidate
    raise FileNotFoundError('无法定位 MoE 实验根目录')

LAB_DIR = locate_lab_dir()

SRC_DIR = LAB_DIR / 'src' / 'moe_sort_lab'
print('LAB_DIR =', LAB_DIR)
print('ASCEND_HOME_PATH =', os.environ.get('ASCEND_HOME_PATH', '<未设置>'))
assert (SRC_DIR / 'scripts' / 'build_ops.sh').is_file()
print('环境与目录检查通过')

In [ ]:
# msopgen 拒绝在含非 ASCII 字符、或 group/other 可写的路径下生成算子工程。
# 若当前实验目录（或其祖先路径）不满足要求，自动整章复制到 /tmp 下的英文
# 目录并修正权限，再继续后续构建；已满足时保持原位，不产生任何复制。
import shutil
import tempfile
from pathlib import Path as _Path

def _msopgen_safe(root):
    try:
        current = _Path(root).resolve()
    except OSError:
        return False
    while True:
        text = str(current)
        if any(ord(ch) > 127 for ch in text):
            return False
        try:
            mode = os.stat(current).st_mode
        except OSError:
            return False
        if current == current.parent:  # 已到文件系统根
            break
        if (mode & 0o022) and not (mode & 0o1000):  # 非 sticky 却 group/other 可写
            return False
        current = current.parent
    return True

if not _msopgen_safe(LAB_DIR):
    reloc = _Path(tempfile.gettempdir()) / f'cannlab_moe_sort_lab_{os.getpid()}'
    if reloc.exists():
        shutil.rmtree(reloc)
    shutil.copytree(
        LAB_DIR, reloc,
        ignore=shutil.ignore_patterns('build', 'generated', 'data',
                                      '__pycache__', '*.pyc', '.git'),
    )
    for root_dir, dirs, files in os.walk(reloc):
        os.chmod(root_dir, 0o755)
        for name in files:
            os.chmod(_Path(root_dir) / name, 0o644)
    os.chdir(reloc)
    LAB_DIR = reloc
    SRC_DIR = LAB_DIR / 'src' / 'moe_sort_lab'
    print(f'检测到 msopgen 不接受的路径，已复制到英文临时目录：{reloc}')
    print('LAB_DIR =', LAB_DIR)
else:
    print('实验路径满足 msopgen 要求，直接原位执行。')

# Windows 工作区中的 CRLF 会使 Bash 将行尾的 \r 当作命令的一部分。
# 无论是否迁移目录，均在执行前将实验脚本统一为 LF。
for shell_file in _Path(LAB_DIR).rglob('*.sh'):
    content = shell_file.read_bytes()
    if b'\r\n' in content:
        shell_file.write_bytes(content.replace(b'\r\n', b'\n'))


## 2. 查看工程结构

工程源码位于 `src/moe_sort_lab`。`custom_ops/src` 保存可复现的 Host/Kernel 源码，构建脚本会根据目标芯片准备算子工程，后续步骤使用该工程完成编译。

In [ ]:
for path in sorted(SRC_DIR.rglob('*')):
    if path.is_file() and 'build' not in path.parts and 'generated' not in path.parts:
        print(path.relative_to(SRC_DIR))

## 3. 阅读关键源码

先看 Host tiling，再看 Kernel。Host 侧根据 `[T, E]` 计算 `tokensPerCore` 并设置 `BLOCK_DIM`；Kernel 通过 `GetBlockIdx()` 取得自己的 token 范围，不直接判断 910B 或 310B。

In [ ]:
files = [
    'custom_ops/src/MoeSortQuickSortLite/op_host/moe_sort_quick_sort_lite.cpp',
    'custom_ops/src/MoeSortQuickSortLite/op_kernel/moe_sort_quick_sort_lite.cpp',
    'custom_ops/src/MoeSortHeapSortLite/op_kernel/moe_sort_heap_sort_lite.cpp',
    'aclnn_runner/main_full_pipeline_benchmark.cpp',
]
for relative in files:
    path = SRC_DIR / relative
    print(f'\n===== {relative} =====')
    print(''.join(path.read_text(encoding='utf-8').splitlines(True)[:45]))

## 4. 为 910B 编译自定义算子

下面命令默认使用 `TARGET=ascend910b`。脚本会调用 `msopgen` 生成工程，复制五个算子的源码，将 Host 侧 `BLOCK_DIM` 设置为 16，并将算子包部署到本实验的独立 OPP 路径。

In [ ]:
import subprocess

build = subprocess.run(
    ['bash', 'scripts/build_ops.sh'],
    cwd=SRC_DIR,
    text=True,
    capture_output=True,
)
print(build.stdout)
print(build.stderr)
# 若失败发生在 "Install custom OPP package" 阶段的 rm -rf/mkdir，
# 通常是旧 local_opp 残留只读文件所致；build_ops.sh 会先恢复写权限再删除，
# 仍失败时再检查 CANN 与 msopgen 环境。
assert build.returncode == 0, '算子构建失败：请查看上方输出定位具体阶段。'


## 5. 编译 ACLNN runner

Runner 负责构造 ACL tensor、调用五个自定义算子并复制结果回 Host。`main_benchmark` 侧重比较三个路由 kernel；`main_full_pipeline_benchmark` 侧重比较 route → sortedOrder → Permute → Unpermute 的完整路径。

In [ ]:
runner = subprocess.run(
    ['bash', 'scripts/build_runner.sh'],
    cwd=SRC_DIR,
    text=True,
    capture_output=True,
)
print(runner.stdout)
print(runner.stderr)
assert runner.returncode == 0, 'runner 构建失败，请检查 custom OPP 安装路径'

## 6. 生成确定性测试数据

数据脚本生成 logits、token、排序顺序、identity expert 输出和 CPU reference。固定 `K=2`，保证三种选择策略可以使用相同输入进行比较。

In [ ]:
data = subprocess.run(
    ['python3', 'scripts/gen_data.py', '--num_tokens', '1024', '--num_experts', '64', '--hidden_size', '128', '--top_k', '2'],
    cwd=SRC_DIR,
    text=True,
    capture_output=True,
)
print(data.stdout)
print(data.stderr)
assert data.returncode == 0

## 7. 比较三种路由选择

该命令只比较 TopK、QuickSort 和 HeapSort 的选择时间，并检查三个结果的 expert indices 是否一致。时间为设备实际运行结果，数值会随 CANN 版本、频率和设备状态变化。

In [ ]:
benchmark = subprocess.run(
    'source scripts/env_custom_opp.sh && aclnn_runner/build/main_benchmark data 1024 128 2',
    cwd=SRC_DIR, shell=True, executable='/bin/bash', text=True, capture_output=True,
)
print(benchmark.stdout)
print(benchmark.stderr)
assert benchmark.returncode == 0

## 8. 比较完整 MoE 路径

完整 benchmark 将 `order+copy` 单独列出：本教学实现没有独立的 device-side BuildSortedOrder 算子，因此该阶段包含 D2H、CPU 排序和 H2D。比较 `end_to_end` 时要同时考虑这部分代价。

In [ ]:
pipeline = subprocess.run(
    'source scripts/env_custom_opp.sh && aclnn_runner/build/main_full_pipeline_benchmark data 1024 128 2',
    cwd=SRC_DIR, shell=True, executable='/bin/bash', text=True, capture_output=True,
)
print(pipeline.stdout)
print(pipeline.stderr)
assert pipeline.returncode == 0

## 9. 结果验证

最后使用 CPU reference 检查三条完整路径的 expert indices 和 Unpermute 输出。完成后进入 `07.03_chapter_test.ipynb`。本 Notebook 只提供验证入口，不预置设备运行结果。

In [ ]:
verify = subprocess.run(
    ['python3', 'scripts/verify_full_pipeline.py', 'data'],
    cwd=SRC_DIR,
    text=True,
    capture_output=True,
)
print(verify.stdout)
print(verify.stderr)
assert verify.returncode == 0